In [0]:
import dlt
import pyspark.sql.functions as F

In [0]:
@dlt.table(name="dim_date", table_properties={"quality": "gold"})
def dim_date():
    start_date = "2020-01-01"
    end_date   = "2030-12-31"

    # one row per day between start and end
    days = spark.sql(
        f"SELECT explode(sequence(to_date('{start_date}'), "
        f"to_date('{end_date}'), interval 1 day)) AS full_date"
    )

    dim = days.select(
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_id"),   # <-- matches your facts
        F.col("full_date"),
        F.year("full_date").alias("year"),
        F.quarter("full_date").alias("quarter"),
        F.month("full_date").alias("month"),
        F.date_format("full_date", "MMMM").alias("month_name"),
        F.dayofmonth("full_date").alias("day"),
        F.dayofweek("full_date").alias("day_of_week"),        # 1=Sun … 7=Sat
        F.date_format("full_date", "EEEE").alias("day_name"),
        F.weekofyear("full_date").alias("week_of_year"),
        F.dayofyear("full_date").alias("day_of_year"),
        F.dayofweek("full_date").isin(1, 7).alias("is_weekend"),
    )

    # optional but recommended: a -1 "unknown date" member for facts with null created_at
    unknown = spark.range(1).select(
        F.lit(-1).cast("int").alias("date_id"),
        F.lit(None).cast("date").alias("full_date"),
        F.lit(None).cast("int").alias("year"),
        F.lit(None).cast("int").alias("quarter"),
        F.lit(None).cast("int").alias("month"),
        F.lit("Unknown").alias("month_name"),
        F.lit(None).cast("int").alias("day"),
        F.lit(None).cast("int").alias("day_of_week"),
        F.lit("Unknown").alias("day_name"),
        F.lit(None).cast("int").alias("week_of_year"),
        F.lit(None).cast("int").alias("day_of_year"),
        F.lit(None).cast("boolean").alias("is_weekend"),
    )

    return dim.unionByName(unknown)